# 🎙️ InterviewCoach — Analysis Pipeline

This notebook is the **offline / reference version** of what the web app does.
Everything runs **locally** (NumPy, librosa, open-source Whisper from Hugging Face) — **no external speech API**.

---
### What changed vs. the first version
* **Pauses are measured from the real audio** (adaptive energy VAD — the same algorithm as `app.py`), not from made-up timestamps.
* **WPM is reported twice**: overall, and *excluding pauses* (speaking rate).
* **Optional local transcription** with Whisper (Hugging Face Transformers + PyTorch), including word timestamps.

### Contents
1. Configuration · 2. Sample transcripts · 3. Analysis functions · 4. Pause detection from audio
5. English report · 6. Optional: transcribe a real recording locally · 7. Indonesian · 8. JSON export

In [1]:
import re
import json
from collections import Counter

import numpy as np
from IPython.display import HTML, display

try:
    import librosa            # only needed to resample real recordings
except ImportError:
    librosa = None

## 1. Configuration

In [2]:
# ── Filler Words ──────────────────────────────────────────────────────────────
FILLERS_EN = [
    "um", "uh", "like", "you know", "basically", "literally", "right",
    "actually", "kind of", "sort of", "honestly", "okay", "i mean", "anyway",
    "you see", "well", "so yeah", "and stuff", "or whatever", "you know what i mean"
]

FILLERS_ID = [
    "ee", "eee", "anu", "gitu", "kan", "tuh", "ya", "jadi", "soalnya", "terus",
    "nah", "intinya", "maksudnya", "gimana", "kayak", "emang", "sebenarnya",
    "begitu", "yah", "loh", "deh", "nih", "sih"
]

# ── Stop Words ────────────────────────────────────────────────────────────────
STOP_WORDS_EN = set([
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "it", "they",
    "them", "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "to", "of", "in", "on", "at",
    "for", "with", "about", "by", "from", "up", "out", "as", "and", "or",
    "but", "if", "then", "than", "so", "yet", "both", "either", "not", "no",
    "that", "this", "these", "those", "what", "which", "who", "when", "where",
    "how", "why", "there", "here", "just", "also", "very", "really", "more",
    "most", "some", "any", "all"
])

STOP_WORDS_ID = set([
    "saya", "aku", "kita", "kami", "kamu", "anda", "dia", "mereka", "nya",
    "yang", "dan", "atau", "tapi", "juga", "ke", "di", "dari", "untuk",
    "dengan", "pada", "ini", "itu", "ada", "tidak", "bisa", "akan", "sudah",
    "belum", "jika", "kalau", "karena", "oleh", "lebih", "banyak", "sangat",
    "saat", "ketika", "selalu", "sering", "pernah", "dalam", "lagi", "harus",
    "perlu", "bagi", "hal", "cara", "agar", "maka", "serta", "maupun",
    "bahwa", "tentang"
])

print('✅ Configuration loaded')
print(f'  EN fillers: {len(FILLERS_EN)} words')
print(f'  ID fillers: {len(FILLERS_ID)} words')

✅ Configuration loaded
  EN fillers: 20 words
  ID fillers: 23 words


## 2. Sample Transcript
Replace this with your own transcript to test the pipeline.

In [3]:
# ── Sample English transcript ─────────────────────────────────────────────────
SAMPLE_TRANSCRIPT_EN = """
So, um, tell me about yourself. Well, I am basically a software engineer with,
you know, about five years of experience. I've worked at, like, three different
companies and, uh, I really enjoy, sort of, building things that help people.
I'm actually quite passionate about, you know, user experience design. I mean,
honestly, I think the best software is software that, um, people actually want
to use. Right? So yeah, that's kind of my background in a nutshell.
""".strip()

# ── Sample Indonesian transcript ──────────────────────────────────────────────
SAMPLE_TRANSCRIPT_ID = """
Jadi, ee, saya adalah seorang pengembang perangkat lunak yang sudah bekerja
selama, ya, sekitar lima tahun. Saya pernah bekerja di, anu, tiga perusahaan
yang berbeda. Intinya, saya sangat suka, gitu, membangun aplikasi yang berguna
bagi pengguna. Maksudnya, saya percaya bahwa teknologi, kan, harus bisa
memudahkan kehidupan. Nah, itu sih gambaran singkat tentang saya.
""".strip()

print('✅ Sample transcripts ready')
print('\n--- English Sample ---')
print(SAMPLE_TRANSCRIPT_EN[:200], '...')

✅ Sample transcripts ready

--- English Sample ---
So, um, tell me about yourself. Well, I am basically a software engineer with,
you know, about five years of experience. I've worked at, like, three different
companies and, uh, I really enjoy, sort o ...


## 3. Text Analysis Functions

In [4]:
def count_fillers(text: str, lang: str = 'en') -> dict:
    """Count each filler word occurrence in the transcript."""
    fillers = FILLERS_EN if lang == 'en' else FILLERS_ID
    lower = text.lower()
    result = {}
    # Sort by length descending to match multi-word fillers first
    for f in sorted(fillers, key=len, reverse=True):
        pattern = re.compile(r'\b' + re.escape(f) + r'\b', re.IGNORECASE)
        matches = pattern.findall(lower)
        if matches:
            result[f] = len(matches)
    return result


def calculate_wpm(text: str, duration_seconds: float) -> float:
    """Calculate words per minute."""
    if duration_seconds <= 0:
        return 0.0
    word_count = len(text.split())
    return round((word_count / duration_seconds) * 60, 1)


def extract_keywords(text: str, lang: str = 'en', top_n: int = 12) -> list:
    """Extract most frequent meaningful words, excluding stop words and fillers."""
    stop_words = STOP_WORDS_EN if lang == 'en' else STOP_WORDS_ID
    fillers = set(FILLERS_EN if lang == 'en' else FILLERS_ID)
    
    words = re.findall(r'\b[a-zA-ZÀ-ÿ]{4,}\b', text.lower())
    filtered = [w for w in words if w not in stop_words and w not in fillers]
    freq = Counter(filtered)
    return freq.most_common(top_n)


def highlight_fillers_html(text: str, lang: str = 'en') -> str:
    """Return HTML string with filler words wrapped in highlight spans."""
    fillers = FILLERS_EN if lang == 'en' else FILLERS_ID
    result = text
    style = 'background:#FFF3E0;color:#E65100;border-radius:4px;padding:1px 5px;font-weight:bold;border-bottom:2px solid #FFCC80;'
    for f in sorted(fillers, key=len, reverse=True):
        pattern = re.compile(r'\b(' + re.escape(f) + r')\b', re.IGNORECASE)
        result = pattern.sub(f'<span style="{style}">\\1</span>', result)
    return result


print('✅ Text analysis functions defined')

✅ Text analysis functions defined


## 4. Pause Detection From Audio

Same algorithm as the web app:

1. Split the 16 kHz signal into **20 ms frames** and take each frame's RMS energy.
2. **Adaptive noise floor** → threshold = `3 × noise` (clamped to 0.009 – 0.06), so it works on quiet and noisy microphones.
3. **≥ 100 ms** of continuous energy counts as speech (clicks / coughs are ignored); dips shorter than **350 ms** stay inside the phrase.
4. A silence **between two speech stretches** that lasts **≥ 1.5 s** is a *pause*. (Silence before the first word and after the last word is not counted.)

In [5]:
SR = 16000
FRAME = 320                    # 20 ms
FRAME_SEC = FRAME / SR
MIN_RUN_FRAMES = 5             # 100 ms of energy = speech
HANGOVER_SEC = 0.35
PAUSE_MIN_SEC = 1.5            # counted as a pause
PAUSE_LONG_SEC = 3.0           # "long pause"


def frame_rms(y, frame=FRAME):
    n = len(y) // frame
    return np.sqrt((y[: n * frame].reshape(n, frame) ** 2).mean(axis=1))


def detect_pauses(y, sr=SR, min_pause=PAUSE_MIN_SEC):
    """Measure pauses from a mono waveform (float, -1..1). Returns a dict."""
    y = np.asarray(y, dtype=np.float32)
    if sr != SR:
        if librosa is None:
            raise RuntimeError('pip install librosa to resample audio')
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)

    noise = 0.003
    voiced_run = 0
    run_start = 0.0
    speaking = ever = False
    last_end = first = 0.0
    pauses = []

    for k, r in enumerate(frame_rms(y)):
        t_end = (k + 1) * FRAME_SEC
        t_start = t_end - FRAME_SEC
        thr = min(0.06, max(0.009, noise * 3))
        if r > thr:
            if voiced_run == 0:
                run_start = t_start
            voiced_run += 1
            if voiced_run >= MIN_RUN_FRAMES:
                if not speaking:
                    speaking = True
                    gap = (run_start - last_end) if ever else 0.0
                    if gap >= min_pause:
                        pauses.append({'time': round(last_end, 1), 'duration': round(gap, 1)})
                    if not ever:
                        ever, first = True, run_start
                last_end = t_end
        else:
            voiced_run = 0
            noise += (r - noise) * (0.2 if r < noise else 0.01)
            noise = min(0.03, max(0.0005, noise))
            if speaking and t_end - last_end > HANGOVER_SEC:
                speaking = False

    span = max(0.0, last_end - first) if ever else 0.0
    paused = sum(p['duration'] for p in pauses)
    return {
        'pauses': pauses,
        'duration': len(y) / SR,
        'first_speech': round(first, 2),
        'last_speech': round(last_end, 2),
        'articulation_sec': round(max(0.0, span - paused), 2),   # time actually talking
    }


def pauses_from_words(words, min_pause=PAUSE_MIN_SEC):
    """Second opinion from Whisper word timestamps: [(word, start, end), ...]"""
    out = []
    for (_, _, e0), (_, s1, _) in zip(words, words[1:]):
        if s1 - e0 >= min_pause:
            out.append({'time': round(e0, 1), 'duration': round(s1 - e0, 1)})
    return out


def calculate_wpm_articulation(text: str, articulation_seconds: float) -> float:
    """Speaking rate: words per minute counting only the time spent talking."""
    return calculate_wpm(text, articulation_seconds)


def make_demo_audio(plan, sr=SR, seed=0):
    """Synthetic 'speech' (harmonics with syllable-like wobble) + silence, so the notebook runs without a recording.
    plan = [('silence'|'speech', seconds), ...]"""
    rng = np.random.default_rng(seed)
    parts = []
    for kind, sec in plan:
        t = np.arange(int(sec * sr)) / sr
        y = rng.uniform(-0.002, 0.002, t.size)
        if kind == 'speech':
            wob = 0.6 + 0.4 * np.abs(np.sin(2 * np.pi * 4 * t))
            y += 0.15 * wob * (np.sin(2 * np.pi * 180 * t) + 0.5 * np.sin(2 * np.pi * 360 * t)) / 1.5
        parts.append(y)
    return np.concatenate(parts).astype(np.float32)


print('✅ Pause detection ready')

✅ Pause detection ready


## 5. Run Analysis — English

In [6]:
# ── Parameters ────────────────────────────────────────────────────────────────
transcript = SAMPLE_TRANSCRIPT_EN
lang = 'en'

# Audio: replace with your own recording →  y, sr = librosa.load('answer.wav', sr=16000, mono=True)
# Demo audio: 3 s speech · 2 s pause · 6 s speech · 4.2 s pause · 7 s speech · 1 s silence at the end
y = make_demo_audio([('silence', 0.8), ('speech', 3.0), ('silence', 2.0), ('speech', 6.0),
                     ('silence', 4.2), ('speech', 7.0), ('silence', 1.0)])
sr = SR

# ── Run ───────────────────────────────────────────────────────────────────────
pa = detect_pauses(y, sr)
pauses = pa['pauses']
duration_seconds = pa['duration']

words = transcript.split()
word_count = len(words)
wpm = calculate_wpm(transcript, duration_seconds)
wpm_speaking = calculate_wpm_articulation(transcript, pa['articulation_sec'])
fillers_found = count_fillers(transcript, lang)
total_fillers = sum(fillers_found.values())
filler_pct = round((total_fillers / word_count) * 100, 1) if word_count > 0 else 0
keywords = extract_keywords(transcript, lang)
pause_total = round(sum(p['duration'] for p in pauses), 1)

# ── Report ────────────────────────────────────────────────────────────────────
print('═' * 55)
print('  📊 INTERVIEWCOACH SESSION REPORT')
print('═' * 55)
mins, secs = int(duration_seconds // 60), int(duration_seconds % 60)
print(f'  ⏱️  Duration:      {mins}:{secs:02d}')
print(f'  💬 Word Count:    {word_count}')
print(f'  📈 WPM:           {wpm}   ({wpm_speaking} excluding pauses)')
print(f'  🔴 Filler Words:  {total_fillers} ({filler_pct}% of words)')
print(f'  ⏸️  Pauses:        {len(pauses)} detected (>{PAUSE_MIN_SEC}s) · total {pause_total}s')
print(f'  🔑 Keywords:      {len(keywords)} found')
print()

if fillers_found:
    print('  🔴 Filler Breakdown:')
    for word, count in sorted(fillers_found.items(), key=lambda x: -x[1]):
        print(f'     {word:<20} {"█" * count} ({count}x)')
    print()

if keywords:
    print('  🔑 Top Keywords:')
    for word, count in keywords:
        print(f'     {word:<20} {count}x')
    print()

if pauses:
    print('  ⏸️  Pauses:')
    for p in pauses:
        kind = 'long //' if p['duration'] >= PAUSE_LONG_SEC else 'short /'
        print(f'     At {int(p["time"] // 60)}:{int(p["time"] % 60):02d} → {p["duration"]}s pause  ({kind})')

print('═' * 55)

═══════════════════════════════════════════════════════
  📊 INTERVIEWCOACH SESSION REPORT
═══════════════════════════════════════════════════════
  ⏱️  Duration:      0:24
  💬 Word Count:    78
  📈 WPM:           195.0   (292.5 excluding pauses)
  🔴 Filler Words:  16 (20.5% of words)
  ⏸️  Pauses:        2 detected (>1.5s) · total 6.2s
  🔑 Keywords:      12 found

  🔴 Filler Breakdown:
     you know             ██ (2x)
     actually             ██ (2x)
     um                   ██ (2x)
     basically            █ (1x)
     honestly             █ (1x)
     kind of              █ (1x)
     sort of              █ (1x)
     so yeah              █ (1x)
     i mean               █ (1x)
     right                █ (1x)
     like                 █ (1x)
     well                 █ (1x)
     uh                   █ (1x)

  🔑 Top Keywords:
     software             3x
     know                 2x
     experience           2x
     people               2x
     tell                 1x
     yourself  

## 5b. Highlighted Transcript (HTML rendering)

In [7]:
highlighted = highlight_fillers_html(transcript, lang)

html_output = f"""
<div style="
  font-family: 'Segoe UI', Arial, sans-serif;
  padding: 20px 24px;
  background: linear-gradient(135deg, #E3F2FD, #ffffff);
  border-radius: 12px;
  border: 1.5px solid #90CAF9;
  font-size: 15px;
  line-height: 1.9;
  color: #0D47A1;
  max-width: 800px;
">
  <div style="font-weight:700; font-size:13px; letter-spacing:1px; text-transform:uppercase;
              color:#1565C0; margin-bottom:12px; display:flex; align-items:center; gap:8px;">
    📄 Final Transcript
    <span style="margin-left:auto; font-size:11px; background:#FFF3E0; border:1px solid #FFCC80;
                border-radius:4px; padding:2px 8px; color:#E65100; font-weight:600;
                text-transform:none; letter-spacing:0;">filler = highlighted</span>
  </div>
  {highlighted}
</div>
"""

display(HTML(html_output))

## 6. Optional — Transcribe a Real Recording Locally (Whisper)

Open-source Whisper weights from Hugging Face, run on your own CPU/GPU (no API).
Set `AUDIO_PATH` to a recording; leave it `None` to skip.

**Accuracy / speed tips**
* English → `openai/whisper-small.en` (or `distil-whisper/distil-small.en`: ~6× faster, nearly the same accuracy). Indonesian → `openai/whisper-small` (`large-v3-turbo` if you have a GPU).
* `pip install faster-whisper` is another fully-local option: CTranslate2 + int8 is ~4× faster than PyTorch on CPU and has a built-in VAD filter.
* Whisper tends to **drop "um / uh"**; word-based filler counts are therefore a lower bound.

In [8]:
AUDIO_PATH = None            # e.g. 'my_answer.wav'
ASR_MODEL = 'openai/whisper-small.en'      # Indonesian: 'openai/whisper-small'
ASR_LANGUAGE = None          # None for *.en models; 'indonesian' for multilingual models

if AUDIO_PATH:
    import torch
    from transformers import pipeline

    y_real, _ = librosa.load(AUDIO_PATH, sr=SR, mono=True)
    asr = pipeline('automatic-speech-recognition', model=ASR_MODEL,
                   device=0 if torch.cuda.is_available() else -1)
    gen = {'language': ASR_LANGUAGE, 'task': 'transcribe'} if ASR_LANGUAGE else {}
    out = asr({'raw': y_real, 'sampling_rate': SR}, return_timestamps='word',
              chunk_length_s=30, generate_kwargs=gen)

    real_words = [(c['text'].strip(), c['timestamp'][0], c['timestamp'][1])
                  for c in out['chunks'] if c['text'].strip() and None not in c['timestamp']]
    real_text = out['text'].strip()
    real_pa = detect_pauses(y_real, SR)

    print('Transcript :', real_text)
    print('Pauses (audio energy)    :', real_pa['pauses'])
    print('Pauses (word timestamps) :', pauses_from_words(real_words))
    print(f"WPM {calculate_wpm(real_text, real_pa['duration'])}  |  excl. pauses "
          f"{calculate_wpm_articulation(real_text, real_pa['articulation_sec'])}")
else:
    print('ℹ️  Set AUDIO_PATH to try a real recording (skipped).')

ℹ️  Set AUDIO_PATH to try a real recording (skipped).


## 7. Run Analysis — Bahasa Indonesia

In [9]:
transcript_id = SAMPLE_TRANSCRIPT_ID
lang_id = 'id'

# demo audio: 2.5 s speech · 3.4 s pause · 8 s speech · 1.6 s pause · 5 s speech
y_id = make_demo_audio([('silence', 0.5), ('speech', 2.5), ('silence', 3.4), ('speech', 8.0),
                        ('silence', 1.6), ('speech', 5.0), ('silence', 0.8)], seed=1)
pa_id = detect_pauses(y_id, SR)
duration_id = pa_id['duration']

words_id = transcript_id.split()
wc_id = len(words_id)
wpm_id = calculate_wpm(transcript_id, duration_id)
wpm_id_speaking = calculate_wpm_articulation(transcript_id, pa_id['articulation_sec'])
fillers_id = count_fillers(transcript_id, lang_id)
total_fillers_id = sum(fillers_id.values())
filler_pct_id = round((total_fillers_id / wc_id) * 100, 1) if wc_id > 0 else 0
keywords_id = extract_keywords(transcript_id, lang_id)

print('═' * 55)
print('  📊 LAPORAN SESI INTERVIEWCOACH (BAHASA INDONESIA)')
print('═' * 55)
mins_id, secs_id = int(duration_id // 60), int(duration_id % 60)
print(f'  ⏱️  Durasi:         {mins_id}:{secs_id:02d}')
print(f'  💬 Jumlah Kata:    {wc_id}')
print(f'  📈 KPM:            {wpm_id}   ({wpm_id_speaking} tanpa jeda)')
print(f'  🔴 Kata Filler:    {total_fillers_id} ({filler_pct_id}% dari total kata)')
print(f'  ⏸️  Jeda:           {len(pa_id["pauses"])} terdeteksi (>{PAUSE_MIN_SEC}s)')
print(f'  🔑 Kata Kunci:     {len(keywords_id)} ditemukan')
print()

if fillers_id:
    print('  🔴 Rincian Filler:')
    for word, count in sorted(fillers_id.items(), key=lambda x: -x[1]):
        print(f'     {word:<20} {"█" * count} ({count}x)')
    print()

for p in pa_id['pauses']:
    print(f'  ⏸️  Pada {p["time"]}s → jeda {p["duration"]}s')

print()
print('  📄 Transkrip dengan Sorotan:')
highlighted_id = highlight_fillers_html(transcript_id, lang_id)
html_id = f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;padding:16px 20px;
  background:linear-gradient(135deg,#E3F2FD,#fff);border-radius:12px;
  border:1.5px solid #90CAF9;font-size:14px;line-height:1.9;color:#0D47A1;max-width:720px;">
  {highlighted_id}
</div>"""
display(HTML(html_id))

═══════════════════════════════════════════════════════
  📊 LAPORAN SESI INTERVIEWCOACH (BAHASA INDONESIA)
═══════════════════════════════════════════════════════
  ⏱️  Durasi:         0:21
  💬 Jumlah Kata:    53
  📈 KPM:            145.9   (205.2 tanpa jeda)
  🔴 Kata Filler:    10 (18.9% dari total kata)
  ⏸️  Jeda:           2 terdeteksi (>1.5s)
  🔑 Kata Kunci:     12 ditemukan

  🔴 Rincian Filler:
     maksudnya            █ (1x)
     intinya              █ (1x)
     gitu                 █ (1x)
     jadi                 █ (1x)
     anu                  █ (1x)
     kan                  █ (1x)
     nah                  █ (1x)
     sih                  █ (1x)
     ee                   █ (1x)
     ya                   █ (1x)

  ⏸️  Pada 3.0s → jeda 3.4s
  ⏸️  Pada 14.4s → jeda 1.6s

  📄 Transkrip dengan Sorotan:


## 8. Export Report as JSON

In [10]:
report = {
    "language": "en",
    "duration_seconds": round(duration_seconds, 1),
    "word_count": word_count,
    "wpm": wpm,
    "wpm_excluding_pauses": wpm_speaking,
    "filler_words": fillers_found,
    "total_fillers": total_fillers,
    "filler_percentage": filler_pct,
    "pauses": pauses,
    "pause_total_seconds": pause_total,
    "keywords": [{'word': w, 'count': c} for w, c in keywords],
    "transcript": transcript
}

print(json.dumps(report, indent=2))

{
  "language": "en",
  "duration_seconds": 24.0,
  "word_count": 78,
  "wpm": 195.0,
  "wpm_excluding_pauses": 292.5,
  "filler_words": {
    "basically": 1,
    "you know": 2,
    "actually": 2,
    "honestly": 1,
    "kind of": 1,
    "sort of": 1,
    "so yeah": 1,
    "i mean": 1,
    "right": 1,
    "like": 1,
    "well": 1,
    "um": 2,
    "uh": 1
  },
  "total_fillers": 16,
  "filler_percentage": 20.5,
  "pauses": [
    {
      "time": 3.8,
      "duration": 2.0
    },
    {
      "time": 11.8,
      "duration": 4.2
    }
  ],
  "pause_total_seconds": 6.2,
  "keywords": [
    {
      "word": "software",
      "count": 3
    },
    {
      "word": "know",
      "count": 2
    },
    {
      "word": "experience",
      "count": 2
    },
    {
      "word": "people",
      "count": 2
    },
    {
      "word": "tell",
      "count": 1
    },
    {
      "word": "yourself",
      "count": 1
    },
    {
      "word": "engineer",
      "count": 1
    },
    {
      "word": "five",


---
## 🚀 How to Run the InterviewCoach Web App

```bash
pip install streamlit
streamlit run app.py        # then open http://localhost:8501
```

The app downloads the open-source Whisper model **once** (Hugging Face), caches it in the browser, and from then on
transcribes **on your device** — with WebGPU when available, otherwise on the CPU (WASM).
Only for this notebook: `pip install numpy librosa` (and `transformers torch` for section 6).